#  Last Mile Delivery — Mini Hackathon Analysis
**Dataset:** 2,080 delivery orders across 10 Indian cities (full year 2024)  
**Questions covered:** Q1 Peak-hour delays · Q2 Weather EDA · Q3 Rider experience (stats) · Q4 City dashboard

---

## 0. Setup — Imports & Load

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from scipy import stats
import warnings; warnings.filterwarnings('ignore')

df = pd.read_csv('last_mile_delivery_dataset.csv')
print(f'Shape: {df.shape}')
df.head(3)

## 1. Data Cleaning
Key issues found:
- **vehicle_type**: mixed case (bike / Bike / BIKE → Bike)
- **city**: typos and extra spaces (Hydrabad, Bangaluru, 'Kolkata ', ' Ahmedabad')
- **zone / rider_id**: ~6–7% null → filled with 'Unknown'
- **gps_latitude/longitude**: 137 nulls — not used in analysis
- Derived columns: `hour`, `month`, `peak` flag


In [ ]:
# Fix vehicle_type casing
df['vehicle_type'] = df['vehicle_type'].str.strip().str.title()

# Fix city names
city_map = {'delhi':'Delhi','MUMBAI':'Mumbai','chennai':'Chennai',
            ' Ahmedabad':'Ahmedabad','Bangaluru':'Bangalore',
            'Kolkata ':'Kolkata','Hydrabad':'Hyderabad'}
df['city'] = df['city'].str.strip().replace(city_map)

# Fill nulls
df['zone']     = df['zone'].fillna('Unknown')
df['rider_id'] = df['rider_id'].fillna('UNKNOWN')

# Parse datetime features
df['order_date'] = pd.to_datetime(df['order_date'])
df['hour']       = df['order_time'].str[:2].astype(int)
df['month']      = df['order_date'].dt.month

print('Cities:', sorted(df['city'].unique()))
print('Vehicle types:', sorted(df['vehicle_type'].unique()))
print('Nulls:', df.isnull().sum()[df.isnull().sum()>0].to_dict())

## Q1 — Peak Hour Delay Pattern
> Do orders placed during **8–10 AM** and **5–8 PM** have significantly higher `delay_mins` than off-peak hours?

**Method:** Mann-Whitney U test (non-parametric, since delay distribution is skewed)

In [ ]:
df['peak'] = df['hour'].apply(lambda h: 'Peak' if (8<=h<=10 or 17<=h<=20) else 'Off-Peak')

peak  = df[df['peak']=='Peak']['delay_mins']
offpk = df[df['peak']=='Off-Peak']['delay_mins']

print(f'Peak    : n={len(peak)}   mean={peak.mean():.2f} min   median={peak.median():.2f} min')
print(f'Off-Peak: n={len(offpk)}  mean={offpk.mean():.2f} min   median={offpk.median():.2f} min')
print(f'Delta   : {peak.mean()-offpk.mean():.2f} min ({(peak.mean()-offpk.mean())/offpk.mean()*100:.0f}% higher in peak)')

u, p = stats.mannwhitneyu(peak, offpk, alternative='greater')
print(f'Mann-Whitney U={u:.0f},  p={p:.2e}  → {"SIGNIFICANT" if p<0.05 else "not significant"}')

In [ ]:
from IPython.display import Image
Image('q1_peak_hour_delay.png')

## Q2 — Weather vs Delay (EDA)
> How does median delay vary across **Clear, Rain, Fog**?  
> Which `order_type` is hit hardest by rain?


In [ ]:
wx = df[df['weather_condition'].isin(['Clear','Rain','Fog'])]
print('Median delay by weather:')
print(wx.groupby('weather_condition')['delay_mins'].median())

rain = df[df['weather_condition']=='Rain']
print('\nRain impact by order type (median delay):')
print(rain.groupby('order_type')['delay_mins'].median().sort_values(ascending=False))

In [ ]:
Image('q2_weather_eda.png')

## Q3 — Rider Experience Effect (Statistics)
> Is there a **statistically meaningful** difference in `delay_mins` between riders
> with **<2 years** experience vs **>4 years** experience?

**Hypotheses:**
- H₀: Junior and senior riders have the same delay distribution
- H₁: Junior riders have higher delays (one-tailed)

**Test chosen:** Mann-Whitney U (skewed, non-normal data)

In [ ]:
junior = df[df['rider_experience_yrs'] < 2]['delay_mins']
senior = df[df['rider_experience_yrs'] > 4]['delay_mins']

print(f'Junior (<2yr): n={len(junior)}  mean={junior.mean():.2f}  std={junior.std():.2f}')
print(f'Senior (>4yr): n={len(senior)}  mean={senior.mean():.2f}  std={senior.std():.2f}')

u2, p2 = stats.mannwhitneyu(junior, senior, alternative='greater')
d = (junior.mean()-senior.mean()) / np.sqrt((junior.std()**2+senior.std()**2)/2)

print(f'\nMann-Whitney U={u2:.0f},  p={p2:.4f}')
print(f"Cohen's d = {d:.3f}  ({'negligible' if abs(d)<0.2 else 'small' if abs(d)<0.5 else 'medium'} effect size)")
print('\nConclusion: No statistically significant difference (p=0.717 >> 0.05).')
print('Both groups perform almost identically — experience alone does not predict delay.')

In [ ]:
Image('q3_rider_experience.png')

## Q4 — City-Level Performance Dashboard
> 3-panel dashboard: **city on-time rate · monthly delay trend · vehicle type comparison**


In [ ]:
# City on-time rates
city_ontime = df.groupby('city').apply(lambda x: (x['delivery_status']=='On-Time').mean()*100).round(1).sort_values()
print('City On-Time Rates:')
print(city_ontime)
print(f'\nBest city : {city_ontime.idxmax()} ({city_ontime.max():.1f}%)')
print(f'Worst city: {city_ontime.idxmin()} ({city_ontime.min():.1f}%)')

In [ ]:
Image('q4_dashboard.png')

## Final Insights & Observations

| # | Finding | Impact |
|---|---------|--------|
| 1 | Peak hours (8–10 AM, 5–8 PM) add **+15.87 min** average delay vs off-peak | High |
| 2 | Fog causes the worst median delay (37.9 min); Rain is second (29.4 min) | High |
| 3 | Medicine deliveries suffer most in rain (34.65 min median delay) | Critical |
| 4 | Rider experience has **no statistically significant** impact on delay | Insight |
| 5 | Hyderabad is the best performer (51.6% on-time); Bangalore worst (31.1%) | Operational |
| 6 | Bikes are the fastest vehicle type (12.84 min mean delay) | Actionable |
| 7 | Dec and Oct see the highest average delays in the year | Seasonal |

###  One Biggest Fix
> **Implement peak-hour surge staffing or dynamic routing.** The 15.87-minute delay premium
> during peak windows (8–10 AM and 5–8 PM) accounts for the largest single controllable source
> of lateness. Shifting 20% of deliveries outside these windows or adding surge riders during
> these slots could theoretically cut overall delay by ~30%.
